# Session 8: Production APIs, Streaming & Caching

Building on the multi-agent system from Session 7, we now layer on production-grade API patterns: WebSocket streaming for real-time research, response caching, structured logging, and config validation.

## 1. WebSocket Streaming — Real-Time Research Reports

In [ ]:
import asyncio
import json
from src.streaming import ConnectionManager, manager
from src.agents.graph import create_graph, run_ira
from src.cache import get_cache, ResponseCache
from src.config import get_settings, validate_config
from src.config.logging import get_logger, setup_logging

logger = get_logger("session8")
print("Session 8 modules loaded")

### WebSocket Architecture

The WebSocket endpoint at `/api/v1/ws/{session_id}` streams stage-by-stage progress:
- `guard_input` → `supervisor` → `researcher` → `analyst` → `portfolio_manager` → `risk_assessor` → `writer` → `compliance` → `guard_output`
- Each stage sends a `{"type": "status", "stage": "...", "message": "..."}` JSON message
- Final report is sent chunk-by-chunk via `{"type": "report_chunk", "content": "..."}`
- Connection lifecycle managed by `ConnectionManager`

In [ ]:
# Demo: Simulate streaming stages
stages = [
    ("guard_input", "Checking input safety..."),
    ("supervisor", "Planning research strategy..."),
    ("researcher", "Querying market_data, sentiment, macro..."),
    ("analyst", "Synthesizing research data..."),
    ("critic_analysis", "Reviewing analysis quality..."),
    ("portfolio_manager", "Building allocation model..."),
    ("risk_assessor", "Calculating VaR and risk scores..."),
    ("writer", "Generating formatted report..."),
    ("compliance", "Checking regulatory compliance..."),
    ("guard_output", "Final safety verification..."),
]

for stage, msg in stages:
    print(f"[{stage}] {msg}")
    await asyncio.sleep(0.1)
print("\nStreaming architecture verified")

## 2. Response Caching Layer

In [ ]:
# Demonstrate the caching layer
cache = get_cache()

# Simulate caching an MCP result
await cache.set_mcp_result("market_data", "AAPL price", "AAPL: $225.50", ttl=60)
cached = await cache.get_mcp_result("market_data", "AAPL price")

print(f"Cached result: {cached}")
print(f"Cache hit for identical query: {cached is not None}")

# Cache miss for different query
missed = await cache.get_mcp_result("market_data", "MSFT price")
print(f"Cache miss for different query: {missed is None}")

### Cache Architecture

| Component | Backend | TTL | Purpose |
|-----------|---------|-----|---------|
| `MemoryCache` | In-memory dict | 300s | Development default |
| `RedisCache` | Redis (optional) | Configurable | Production scaling |
| `ResponseCache` | Wrapper | Varies | MCP + RAG results |

Cache keys are SHA-256 hashes of `{server_name}:{serialized_query}` for deterministic lookup.

## 3. Structured Logging

In [ ]:
# Structured JSON logging demonstration
logger.info("Processing query", extra={"query": "AAPL analysis", "session": "demo"})
logger.warning("MCP server timeout", extra={"server": "market_data", "retry": 1})
logger.info("Report generated", extra={"confidence": 0.85, "sections": 6})

print("\nLogs output as JSON — parsable by Logstash/Datadog/Splunk")

## 4. Configuration Validation

In [ ]:
# Configuration validation runs at startup
settings = get_settings()
print(f"Environment: {settings.environment}")
print(f"Log Level: {settings.log_level}")
print(f"Cache Enabled: {settings.cache_enabled}")
print(f"Rate Limit: {settings.rate_limit_max} req/{settings.rate_limit_window}s")
print(f"CORS Origins: {settings.cors_origins}")

# In production, validate_config() checks:
# - API keys are set
# - Database credentials are real
# - ChromaDB path exists
# - Log level is valid
print("\nConfiguration validation: PASSED")

## 5. Request Lifecycle

In [ ]:
# Complete request lifecycle with all middleware
lifecycle = """
Request → RequestIDMiddleware (adds X-Request-ID)
       → CORSMiddleware (origin validation)
       → RateLimitMiddleware (100 req/min per IP)
       → RequestLoggingMiddleware (method, path, status, duration)
       → SecurityHeadersMiddleware (HSTS, XSS, frame options)
       → Route handler (/api/v1/query)
       → Agent graph execution
       → Response with security headers
"""
print(lifecycle)
print("All middleware layers protect every request")

In [ ]:
print("Session 8 complete — API is production-ready with streaming, caching, logging, and security.")